# Seven-Arm Dependency-Aware Selective Regeneration Benchmark

This notebook runs the full benchmark on Kaggle with Qwen2.5-Coder.

**Smoke profile** (default): 1 scenario, 7 strategies, non-publication evidence.
**Pilot profile** (requires `--profile pilot`): 12 scenarios, 2 strategies, 2 reps, descriptive only.
**Research profile** (requires `--profile research`): 24 scenarios, 4 strategies, 3 reps, publication.

Only the smoke profile runs by default. Pilot and research require explicit `--profile` selection.

In [ ]:
# Cell 1: Install dependencies
!pip install -r requirements-kaggle.txt
!pip install pyyaml pydantic pytest

In [ ]:
# Cell 2: Verify GPU availability
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU memory: {torch.cuda.get_device_properties(0).total_mem / 1024**3:.1f} GB")
else:
    print("WARNING: No GPU available. Benchmark will run on CPU.")

In [ ]:
# Cell 3: Verify Qwen model is available via Kaggle Dataset mount
from pathlib import Path
import os

kaggle_input = Path("/kaggle/input")
if kaggle_input.is_dir():
    print("Kaggle input directory contents:")
    for item in sorted(kaggle_input.iterdir()):
        print(f"  {item.name}")
else:
    print("WARNING: /kaggle/input not found. Model dataset may not be mounted.")

In [ ]:
# Cell 4: Clone repository and setup
!git clone https://github.com/AhmedEhabH/dependency-aware-selective-regeneration-benchmark.git benchmark
%cd benchmark/project

In [ ]:
# Cell 5: Dry-run smoke validation (orchestration check, no API calls)
import subprocess
import sys

result = subprocess.run(
    [sys.executable, "seven_arm_benchmark.py", "--dry-run", "--profile", "smoke"],
    capture_output=True,
    text=True,
)
print("STDOUT:", result.stdout)
if result.stderr:
    print("STDERR:", result.stderr)
print(f"Return code: {result.returncode}")

In [ ]:
# Cell 6: Real smoke execution (non-publication evidence)
# WARNING: This requires Qwen model to be mounted and GPU available
#
# Smoke is the default profile. Results are marked non-publication.
# Use --profile pilot or --profile research only when explicitly intended.
#
# result = subprocess.run(
#     [sys.executable, "seven_arm_benchmark.py", "--profile", "smoke"],
#     capture_output=True,
#     text=True,
# )
# print("STDOUT:", result.stdout)
# if result.stderr:
#     print("STDERR:", result.stderr)
# print(f"Return code: {result.returncode}")

In [ ]:
# Cell 7: View results
from pathlib import Path
import json

summary_path = Path("runs/benchmark_summary.json")
if summary_path.exists():
    summary = json.loads(summary_path.read_text())
    meta = summary.get("_meta", {})
    print(f"Profile: {meta.get('label', 'unknown')}")
    print(f"Publication evidence: {meta.get('publication_evidence', False)}")
    for arm_name, arm_data in summary.items():
        if arm_name.startswith("_"):
            continue
        print(f"\nArm: {arm_name}")
        print(f"  Success: {arm_data['success_count']}")
        print(f"  Failure: {arm_data['failure_count']}")
        print(f"  Timeout: {arm_data['timeout_count']}")
        print(f"  Duration: {arm_data['total_duration']:.1f}s")
else:
    print("No benchmark summary found. Run the benchmark first.")

## Next Steps

1. **Smoke**: Default profile (non-publication). Runs 1 scenario x 7 strategies.
2. **Pilot**: Requires `--profile pilot`. 12 scenarios, 2 strategies, 2 reps. Descriptive only.
3. **Research**: Requires `--profile research`. 24 scenarios, 4 strategies, 3 reps. Publication.

Only smoke runs by default. Pilot and research must be explicitly selected.
Each Kaggle session has a 9-hour limit. Large profiles may need multiple sessions.